# Lab E — AMD CPU + GPU Heterogeneous Pipeline / CPU+GPU 异步流水线
## Slide 49–50 · 让 CPU 和 GPU 同时干活

**What will you learn? / 本节学什么？**
- 串行处理的等待在哪里 Where serial processing wastes time waiting
- CPU 预处理与 GPU 推理重叠的异步流水线如何工作 How an async pipeline overlaps CPU preprocessing with GPU inference
- 为什么吞吐可能提高、单张延迟却未必下降 Why throughput can rise while per-image latency may not

**One key idea / 一个核心原理**
> 串行时 GPU 在算第 i 张图、CPU 在空闲；流水线让 CPU 趁 GPU 计算时准备第 i+1 张图——等待被“藏”起来了。Serially the CPU idles while the GPU computes; > pipelining hides that wait by preparing image i+1 during GPU compute of i.

In [ ]:
from pathlib import Path
import sys
REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
print("repo:", REPO)

## 1. The idea / 原理
两条路径使用**完全相同**的检测逻辑（读取 → letterbox 预处理 → YOLO26l 前向 → 后处理），处理**同一批图片**：
- **串行 Serial**：一张图完整走完全部步骤，再处理下一张。
- **异步流水线 Async pipeline**：CPU 线程准备第 i+1 张图的同时，GPU 线程计算第 i 张图（有界队列实现双缓冲）。

Both paths share ONE detection pipeline (load → letterbox → forward → post) on the same images. Serial runs stages one-at-a-time; the async pipeline overlaps CPU prepare of image i+1 with GPU compute of image i (bounded queue = double buffering).

In [ ]:
from course.benchmark import load_bench_images
from course.pipeline import run_serial, run_pipeline, compare_results

STUDENT_N_IMAGES = 60                 # <-- student-adjustable (up to 120)
STUDENT_QUEUE_DEPTH = 4                # <-- student-adjustable double-buffer depth

images = load_bench_images(STUDENT_N_IMAGES)
print(f"{len(images)} images ready (official YOLO26l on W7900 for both modes)")

## 2. Run the baseline: serial / 基线：串行处理

In [ ]:
serial = run_serial(images, imgsz=640)
print(f"serial   : total {serial['total_ms']/1000:.1f} s · "
      f"{serial['images_per_s']:.1f} img/s · e2e P50 {serial['e2e_p50_ms']:.0f} ms")

## 3. Change one thing: async CPU+GPU pipeline / 只改一件事：异步流水线
线程、队列与同步都封装在课程工具模块里，我们只调 `run_pipeline()`。
Threads/queues/sync are hidden in the course module — we just call `run_pipeline()`.

In [ ]:
pipeline = run_pipeline(images, imgsz=640, queue_depth=STUDENT_QUEUE_DEPTH)
print(f"pipeline : total {pipeline['total_ms']/1000:.1f} s · "
      f"{pipeline['images_per_s']:.1f} img/s · e2e P50 {pipeline['e2e_p50_ms']:.0f} ms")

## 4. Compare the results — TWO latency scenarios / 对比：两种延迟场景

**场景 1（本实验）**：100 张图片**突发提交**，比较总吞吐与总耗时——流水线赢在把 CPU 准备与 GPU 计算**重叠**。
**Scenario 1 (this lab)**: a burst of images submitted together — compare total throughput; the pipeline wins by *overlapping* CPU prepare with GPU compute.

**场景 2（真实服务）**：请求**陆续到达**，用户关心**自己这一张**的等待时间——突发队列里，排在后面的请求 e2e 包含排队等待，可能远高于串行的单张执行时间。
**Scenario 2 (real service)**: requests arrive over time and each user cares about THEIR image — under a full burst, queued requests include waiting time and can be much slower than serial per-image time.

所以下面表格里 serial 的 e2e P50 是**单张执行时间（无队列）**，pipeline 的 e2e P50 是**突发提交到出结果（含排队）**——两者**口径不同，不可直接相减**。
So serial e2e P50 is *per-image execution (no queue)* while pipeline e2e P50 is *submit→result under burst (queue included)* — different scopes, never subtract them directly.

除速度外，还验证两条路径**逐目标一致**（同样的图 + 同样的逻辑 → 每个框的类别/坐标/置信度一致，IoU ≥ 0.95）。
Besides speed, per-DETECTION parity is verified (same images + same logic → identical class/box/conf per detection, IoU ≥ 0.95).


In [ ]:
import pandas as pd
cmp = compare_results(serial, pipeline)
display(pd.DataFrame([
    {"mode": "Serial 串行", "total (s)": round(serial['total_ms']/1000, 1),
     "img/s": round(serial['images_per_s'], 1),
     "e2e P50 (ms)": round(serial['e2e_p50_ms'], 0),
     "detections consistent": cmp['consistent']},
    {"mode": "Async pipeline 流水线", "total (s)": round(pipeline['total_ms']/1000, 1),
     "img/s": round(pipeline['images_per_s'], 1),
     "e2e P50 (ms)": round(pipeline['e2e_p50_ms'], 0),
     "detections consistent": cmp['consistent']}]))
print(f"throughput gain: {cmp['throughput_gain_x']:.2f}x · "
      f"mismatched images: {len(cmp['detection_mismatches'])}")

In [ ]:
from course.charts import chart_pipeline
chart_pipeline(serial, pipeline)

## 5b. The REAL timeline / 真实时间轴（不是示意图）

下图用**实测阶段记录**画出两种模式的时间轴（每张图一行：读取+预处理 → GPU 推理 → 后处理）。
如果 CPU 阶段与 GPU 阶段真的重叠，你会看到它们在时间上**交错**，而不是首尾相接。
This chart uses MEASURED per-image stage records (each row: load+preprocess → GPU infer → postprocess). Real overlap shows as time-interleaved stages, not an idealized diagram.


In [ ]:
# Real stage timeline from measured records / 用实测记录画时间轴
import matplotlib.pyplot as plt

def timeline(rec, mode, ax):
    rows = rec["per_image"][:25]
    for j, r in enumerate(rows):
        t0 = sum(r[k] for k in ("load_ms", "pre_ms", "infer_ms", "post_ms"))
        start = t0 - r["load_ms"] - r["pre_ms"] - r["infer_ms"] - r["post_ms"] +                 (r["e2e_ms"] - t0) if mode == "async_pipeline" else 0
        # reconstruct relative times within each image's service window
        base = max(0.0, r["e2e_ms"] - t0) if mode == "async_pipeline" else 0.0
        x_pre = base
        x_inf = base + r["load_ms"] + r["pre_ms"]
        x_pos = x_inf + r["infer_ms"]
        ax.barh(j, r["load_ms"] + r["pre_ms"], left=x_pre, height=0.7,
                color="#4d9fff")                       # CPU load+preprocess
        ax.barh(j, r["infer_ms"], left=x_inf, height=0.7, color="#ff4d5e")
        ax.barh(j, r["post_ms"], left=x_pos, height=0.7, color="#ffb000")
    ax.set_ylabel("image #")
    ax.set_xlabel("time since submit (ms)")
    ax.set_title(f"{mode} — measured stages\n"
                 "blue=CPU load+pre · red=GPU infer · orange=post")

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=False)
timeline(serial, "serial", axes[0])
timeline(pipeline, "async_pipeline", axes[1])
axes[0].legend(handles=[plt.Rectangle((0,0),1,1,color="#4d9fff", label="CPU load+pre / CPU读取+预处理"),
                        plt.Rectangle((0,0),1,1,color="#ff4d5e", label="GPU infer / GPU推理"),
                        plt.Rectangle((0,0),1,1,color="#ffb000", label="postprocess / 后处理")],
           loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()

pre_p50, inf_p50 = serial["pre_p50_ms"], serial["infer_p50_ms"]
print(f"bottleneck check: CPU load+pre P50 = {pre_p50:.1f} ms vs "
      f"GPU infer P50 = {inf_p50:.1f} ms -> "
      f"{'CPU' if pre_p50 > inf_p50 else 'GPU'} is the serial bottleneck")


**回答这些问题 / Answer these**:
1. 哪些阶段能够重叠？重叠了多少？（看上图红蓝交错程度）
   *Which stages overlap, and by how much? (see the interleaving above)*
2. 哪个阶段是瓶颈？如果 CPU 预处理比 GPU 推理更慢，流水线还有意义吗？
   *Which stage is the bottleneck? If CPU preprocessing were slower than GPU inference, would pipelining still help?*
3. 为什么吞吐可能提高、单请求延迟却可能变差？
   *Why can throughput rise while per-request latency gets worse?*
4. 什么情况下异构协同不值得用？（提示：CPU 阶段 ≈ 0 时，收益上限是多少？）
   *When is a CPU+GPU pipeline NOT worth it? (hint: if the CPU stage were ~0, what is the maximum gain?)*


## 5. Why? / 为什么会这样？
- 吞吐提高：GPU 不再等 CPU 读图/预处理，总时间 ≈ max(CPU 阶段, GPU 阶段) × N，而不是两者相加。
- 单张延迟未必下降：每张图仍要走完所有阶段，甚至多了入队/唤醒开销；如果 CPU 预处理比 GPU 推理慢，瓶颈就转移到 CPU——收益由**端到端实测**决定。

- Throughput rises because the GPU no longer waits for CPU work: total time ≈ max(CPU stage, GPU stage) × N instead of their sum.
- Per-image latency may NOT drop: every image still passes all stages (plus queueing). If CPU preprocessing is slower than GPU inference, the bottleneck moves to the CPU. Gains are decided by end-to-end measurement.

## 6. Try it yourself / 学生练习
把 `STUDENT_QUEUE_DEPTH` 改成 `1` 或 `8`（或把 `STUDENT_N_IMAGES` 改成 `120`），**重新运行全部格子**，解释吞吐、两种 e2e 口径和重叠程度的变化。瓶颈在 CPU 还是 GPU？（提示：看时间轴与 `pre_p50` / `infer_p50`。）
Set `STUDENT_QUEUE_DEPTH` to 1 or 8 (or `STUDENT_N_IMAGES` to 120), RE-RUN all cells, and explain the changes in throughput, both e2e scopes and the overlap. Where is the bottleneck? (Hint: the timeline and `pre_p50` / `infer_p50`.)


**Takeaway / 一句话总结**
> 让 CPU 与 GPU 合理协同可以隐藏等待、提高吞吐——但收益大小取决于瓶颈在哪，必须端到端实测。
> Overlapping CPU and GPU work hides waiting and can boost throughput — > but the gain depends on where the bottleneck is, and must be proven > end-to-end.

## Common mistakes / 常见错误
- 把"用了异步队列"说成"CPU/GPU 重叠加速"：重叠必须用时间轴/阶段记录证明（上面第 5b 格）。
  *Using an async queue ≠ proving overlap* — overlap must be shown with stage timelines (cell 5b).
- 把突发排队延迟与串行执行延迟直接相减：口径不同（场景 1 vs 场景 2）。
  *Never subtract burst-queue e2e from serial e2e* — different measurement scopes.
- 只比检测数量不比检测内容：数量相同不代表框/类别/置信度相同（本实验逐目标比对，IoU ≥ 0.95）。
  *Equal detection counts ≠ equal detections* — we compare every box/class/confidence (IoU ≥ 0.95).
